# Week 1 — Foundations: Language, Probability, and Information

Before we tokenize a single string, we anchor NLP in three disciplines: theoretical linguistics, probability theory, and information theory. This week answers *what* language is to a machine and *why* probabilistic modeling is the right framework.

## Learning Objectives

By the end of this week, you will be able to:

- Distinguish morphology, syntax, semantics, and pragmatics, and identify where each shows up in modern NLP systems.
- Derive entropy, cross-entropy, KL divergence, and mutual information from first principles.
- Compute the empirical entropy of English on a real corpus and interpret the result in light of Shannon (1948).
- Set up a reproducible Python environment for the remaining 11 weeks.

## Required Reading

- Shannon, C. E. (1948). *A Mathematical Theory of Communication*.
- Manning, C. D., & Schütze, H. (1999). *Foundations of Statistical NLP*, Chapters 1–2.
- Jurafsky, D., & Martin, J. H. (2024). *Speech and Language Processing*, 3rd ed., Chapter 1.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)

## 1. What is language, to a machine?

Language is hierarchical (phoneme → morpheme → word → phrase → sentence → discourse), generative (finite rules produce infinite utterances), and pervasively ambiguous. A machine sees only a byte stream; every level of structure above that we must model explicitly.

Consider the canonical ambiguous sentence:

> *Time flies like an arrow; fruit flies like a banana.*

The lexeme **flies** is a verb in the first clause and a noun in the second; **like** is a preposition in the first and a verb in the second. No bag-of-characters statistic resolves this — we need syntactic and semantic context. The remainder of the course is, in one sense, a story of building progressively richer context windows.

In [ ]:
sentence = "Time flies like an arrow; fruit flies like a banana."
analysis = {
    'flies (clause 1)': ('verb',  'to move through the air'),
    'flies (clause 2)': ('noun',  'small insects of the order Diptera'),
    'like (clause 1)':  ('prep',  'in the manner of'),
    'like (clause 2)':  ('verb',  'to enjoy / be fond of'),
}
for token, (pos, gloss) in analysis.items():
    print(f"{token:<22} {pos:<6} {gloss}")

## 2. Probability: the language of uncertainty

We use four constructs constantly:

**Conditional probability.**  $P(A \mid B) = P(A, B) / P(B)$.

**Chain rule.** For a sequence of tokens $w_1, \ldots, w_T$:

$$P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_{<t}).$$

This identity is the backbone of every language model in the course.

**Bayes' theorem.** $P(H \mid D) \propto P(D \mid H) P(H)$.

**Maximum likelihood.** Choose $\theta$ that maximizes $\prod_i P(x_i \mid \theta)$, equivalently the log-likelihood $\sum_i \log P(x_i \mid \theta)$.

In [ ]:
# A tiny chain-rule demo on a 3-word sequence with a toy distribution.
# Joint = P(w1) * P(w2|w1) * P(w3|w1,w2)
p_w1     = {'the': 0.6, 'a': 0.4}
p_w2_w1  = {'the': {'cat': 0.5, 'dog': 0.5},
            'a':   {'cat': 0.7, 'dog': 0.3}}
p_w3_w12 = {('the','cat'): {'sat': 0.8, 'ran': 0.2},
            ('the','dog'): {'sat': 0.3, 'ran': 0.7},
            ('a','cat'):   {'sat': 0.6, 'ran': 0.4},
            ('a','dog'):   {'sat': 0.2, 'ran': 0.8}}

def joint(w1, w2, w3):
    return p_w1[w1] * p_w2_w1[w1][w2] * p_w3_w12[(w1,w2)][w3]

total = sum(joint(a,b,c) for a in p_w1 for b in p_w2_w1[a] for c in p_w3_w12[(a,b)])
print(f"P(the,cat,sat) = {joint('the','cat','sat'):.4f}")
print(f"Sum over all sequences = {total:.6f}  (should be 1.0)")

## 3. Information theory: entropy, cross-entropy, KL

**Shannon entropy.**

$$H(p) = -\sum_x p(x) \log p(x).$$

Entropy is the average surprisal of a sample from $p$ — the minimum average number of bits (if log base 2) or nats (if natural log) needed to encode it under an optimal code.

**Cross-entropy** between true $p$ and model $q$:

$$H(p, q) = -\sum_x p(x) \log q(x).$$

**KL divergence:**

$$D_{\mathrm{KL}}(p \,\|\, q) = H(p, q) - H(p) = \sum_x p(x) \log \frac{p(x)}{q(x)} \geq 0.$$

Training a language model with cross-entropy loss is exactly equivalent to minimizing $D_{\mathrm{KL}}(p_{\mathrm{data}} \,\|\, p_\theta)$, because $H(p_{\mathrm{data}})$ does not depend on $\theta$. We will return to this identity in every subsequent week.

In [ ]:
def entropy(p, base=2, eps=1e-12):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log(p + eps)) / np.log(base)

def cross_entropy(p, q, base=2, eps=1e-12):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    return -np.sum(p * np.log(q + eps)) / np.log(base)

def kl_divergence(p, q, base=2, eps=1e-12):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    return np.sum(p[mask] * np.log((p[mask] + eps) / (q[mask] + eps))) / np.log(base)

# Sanity checks
p = np.array([0.25, 0.25, 0.25, 0.25])      # uniform on 4 symbols
q = np.array([0.5, 0.25, 0.125, 0.125])      # skewed
print(f"H(p)        = {entropy(p):.4f} bits  (expected: log2(4) = 2.0)")
print(f"H(q)        = {entropy(q):.4f} bits  (expected: 1.75)")
print(f"H(p, q)     = {cross_entropy(p, q):.4f} bits")
print(f"KL(p || q)  = {kl_divergence(p, q):.4f} bits")
print(f"KL(p || p)  = {kl_divergence(p, p):.4f} bits  (must be 0)")

## 4. Empirical entropy of English

Shannon (1951) estimated English at roughly 1.0–1.5 bits per character — far below the 4.76 bits implied by a uniform distribution over 27 symbols. The gap is what compression algorithms exploit. We reproduce a unigram estimate here; richer n-gram and neural estimates appear in Weeks 5–9.

In [ ]:
# A small English sample. In a real course, swap in WikiText-103 or Brown.
TEXT = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
    "How vexingly quick daft zebras jump! "
    "Sphinx of black quartz, judge my vow. "
    "The five boxing wizards jump quickly. " * 50
).lower()

# Character-level unigram entropy
char_counts = Counter(c for c in TEXT if c.isalpha() or c == ' ')
n_chars = sum(char_counts.values())
char_probs = np.array([c / n_chars for c in char_counts.values()])
H_char = entropy(char_probs)
print(f"Vocabulary size      : {len(char_counts)}")
print(f"Unigram H per char   : {H_char:.3f} bits")
print(f"Uniform baseline     : {np.log2(len(char_counts)):.3f} bits")
print(f"Shannon's estimate   : 1.0 – 1.5 bits per char (with full context)")
print()
print("Most frequent characters:")
for ch, c in char_counts.most_common(8):
    print(f"  {ch!r:5} {c:6d}  p={c/n_chars:.4f}")

In [ ]:
# Word-level unigram entropy
words = TEXT.split()
word_counts = Counter(words)
word_probs = np.array([c / len(words) for c in word_counts.values()])
H_word = entropy(word_probs)
print(f"Vocabulary size (words): {len(word_counts)}")
print(f"Unigram H per word     : {H_word:.3f} bits")
print(f"Uniform baseline       : {np.log2(len(word_counts)):.3f} bits")

# Visualize the Zipf curve — a recurring motif in NLP.
freqs = sorted(word_counts.values(), reverse=True)
ranks = np.arange(1, len(freqs) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].loglog(ranks, freqs, marker='o', linestyle='-')
ax[0].set(xlabel='rank (log)', ylabel='frequency (log)', title='Zipf plot — word frequency vs. rank')
ax[1].bar(range(min(10, len(char_counts))), [c for _, c in char_counts.most_common(10)])
ax[1].set_xticks(range(min(10, len(char_counts))))
ax[1].set_xticklabels([ch if ch != ' ' else '␣' for ch, _ in char_counts.most_common(10)])
ax[1].set(title='Top-10 character frequencies', ylabel='count')
plt.tight_layout(); plt.show()

## 5. Mutual information

For two random variables $X, Y$:

$$I(X; Y) = \sum_{x, y} p(x, y) \log \frac{p(x, y)}{p(x) p(y)} = D_{\mathrm{KL}}(p(x, y) \,\|\, p(x) p(y)).$$

$I(X; Y) = 0$ iff $X \perp Y$. In NLP, the **pointwise mutual information** (PMI) of a (word, context) pair is the foundation of distributional semantics — we will see in Week 4 that word2vec implicitly factorizes a shifted PMI matrix (Levy & Goldberg, 2014).

In [ ]:
def pmi(p_xy, p_x, p_y, eps=1e-12):
    return np.log2((p_xy + eps) / (p_x * p_y + eps))

# Toy: two coins with correlation r
def joint_correlated(r):
    # P(00) = P(11) = (1+r)/4, P(01) = P(10) = (1-r)/4
    p = np.array([[(1+r)/4, (1-r)/4],
                  [(1-r)/4, (1+r)/4]])
    return p

for r in [0.0, 0.5, 0.9]:
    p_xy = joint_correlated(r)
    p_x = p_xy.sum(axis=1)
    p_y = p_xy.sum(axis=0)
    I = sum(p_xy[i,j] * pmi(p_xy[i,j], p_x[i], p_y[j])
            for i in range(2) for j in range(2))
    print(f"correlation r = {r:.1f}  ->  I(X;Y) = {I:.4f} bits")

## 6. Exercises

1. **Derive KL from entropy and cross-entropy.** Show $D_{\mathrm{KL}}(p \| q) = H(p, q) - H(p)$ directly from definitions.
2. **Maximum likelihood ⇔ cross-entropy minimization.** Show that for empirical distribution $\hat p$, minimizing $H(\hat p, q_\theta)$ over $\theta$ is equivalent to maximizing the data log-likelihood $\sum_i \log q_\theta(x_i)$.
3. **Turkish vs. English entropy.** Turkish is agglutinative — a single word can encode what English needs a phrase for. Predict whether character-level entropy is higher or lower for Turkish, then verify on a small corpus.
4. **Asymmetry of KL.** Construct $p, q$ with $D_{\mathrm{KL}}(p \| q) \neq D_{\mathrm{KL}}(q \| p)$. Explain why this asymmetry has implications for variational inference (Week 9–10).

In [ ]:
# Starter for exercise 3: estimate H(English) - H(Turkish) on character level.
ENGLISH = TEXT
TURKISH = ("İnsan hakları evrenseldir. " * 50 +
           "Türkçe sondan eklemeli bir dildir; "
           "tek bir kelime tüm bir cümleye karşılık gelebilir. " * 50).lower()

def char_entropy(s):
    counts = Counter(c for c in s if c.isalpha() or c == ' ')
    n = sum(counts.values())
    probs = np.array([c / n for c in counts.values()])
    return entropy(probs), len(counts)

H_en, V_en = char_entropy(ENGLISH)
H_tr, V_tr = char_entropy(TURKISH)
print(f"English: H = {H_en:.3f} bits, |V| = {V_en}")
print(f"Turkish: H = {H_tr:.3f} bits, |V| = {V_tr}")
print("\nNote: Turkish uses additional letters (ç, ğ, ı, ö, ş, ü), which inflates |V|.")
print("Whether H rises depends on how uniformly those letters are used.")

---

## Next Week

Week 2 — Text Processing and Tokenization. We move from characters to tokens, building BPE, WordPiece, and Unigram LM from scratch.